# Multilingual LVC dataset — German

Adapted from **build_LVC_sentences_short_context_French_v4.ipynb**.

Run this notebook **from the folder containing `DE_normalized_inputs/`**. It generates:

1. Active sentences with pronominal subjects (default: third-person plural *sie*, comparable to French *ils*).
2. Personal passive sentences, excluding *haben* and the oblique *Rache nehmen* template.
3. Active sentences with nominal subjects, using inflected nominative determiners and verb forms.

The notebook uses only the Python standard library and retains the French column naming for generated sentence metadata where possible. Unlike French, **German does not use a comma after the initial time phrase** and finite verbs must occupy second position (V2). The non-finite verb component is final in these main clauses.

### Input normalization and coverage

- `collocation_items_DE_clean_utf8.csv`: 292 rows; fixes *eine Kuss geben* → *einen Kuss geben*.
- `sentence_buildersDE_standardized_utf8.csv`: 180 rows; UTF-8, disambiguated duplicate plural-subject columns, two corrected subject genders.
- `german_verbs_inflection_DE_utf8.csv`: 45 rows; singular/plural *sie* (and formal *Sie*) disambiguated.

The defaults use the first six time phrases and five conservative adverbs to keep the demo manageable; set `MAX_TIME_SPECS = None` and `MAX_ADVERBS = None` for larger experiments. If you also want **all 42 adverbs**, set `ADVERB_SELECTION = None`. Complete exhaustive generation can produce millions of rows, particularly with nominal subjects. The nominal cell preserves the French v4 default of a 10-row random output sample.

The source item rows are preserved in the outputs. See README for documented normalization and limitations.


## 0. Configuration, file loading and German helper functions

Edit the uppercase settings in the next cell before running any generators.

In [ ]:
"""German LVC sentence generator, adapted from the French v4 notebook.

No third-party dependencies. Works with the three normalized UTF-8 CSV files
in DE_normalized_inputs/. Each function writes UTF-8-SIG CSVs that can be
opened in spreadsheet editors or read by Python.

German-specific handling:
* verb-second word order after fronted time phrases (no comma);
* finite auxiliary vs. final participle/infinitive cluster;
* disambiguated sie_sg/sie_pl inflection rows;
* accusative -> nominative NP conversion for ordinary personal passives;
* conservative passive exclusions for haben and oblique-object templates;
* nominative determiner variants for nominal subjects;
* adjective-derived nominal subjects (Angestellter, Reisende, etc.).
"""
from __future__ import annotations
import csv
import json
import random
import re
from collections import Counter
from pathlib import Path

INPUT_DIR = Path('DE_normalized_inputs')
OUTPUT_DIR = Path('DE_generated_sentences')
ITEMS_NAME = 'collocation_items_DE_clean_utf8.csv'
BUILDER_NAME = 'sentence_buildersDE_standardized_utf8.csv'
VERBS_NAME = 'german_verbs_inflection_DE_utf8.csv'
SOURCE_FIELDS = ['LF', 'verb_en', 'keyverb_eng', 'verb_de', 'object_de',
                 'gen_object_de', 'num_obj_de', 'collocation', 'full_expression']
TENSES = ['future', 'present_perfect', 'past_perfect']
TENSE_COL = {
    'future': 'future_active',
    'present_perfect': 'present_perfect_active',
    'past_perfect': 'past_perfect_active',
}
PASSIVE_TENSE_COL = {
    'future': 'future_passive',
    'present_perfect': 'present_perfect_passive',
    'past_perfect': 'past_perfect_passive',
}
# These are intentionally conservative, high-frequency sentential modifiers.
# Set ADVERB_SELECTION = None to use every nonempty builder adverb.
ADVERB_SELECTION = ['anscheinend', 'leider', 'offenbar', 'tatsächlich', 'wahrscheinlich']
MAX_TIME_SPECS = 6   # Set None for the full list of 161 expressions.
MAX_ADVERBS = 5      # Set None to use all selected adverbs.
ACTIVE_PRONOUNS = ['sie_pl']  # German analogue of French v4's ['ils'].
RANDOM_STATE = 99
PASSIVE_EXCLUDED_VERBS = {'haben'}
# French notebook's nominal-subject defaults:
SUBJECT_NUMBER = 'both'       # 'sg', 'pl', 'both'
SUBJECT_DET_TYPES = ['det']   # 'det', 'det_poss', 'no_det'
USE_PRONOUN_SUBJECTS = False
USE_NOMINAL_SUBJECTS = True
USE_ADVERBS_NOMINAL = True
NOMINAL_TENSES = ['present_perfect']
RANDOM_SUBJECT_SELECTION = True
RANDOM_OUTPUT_SAMPLE = True
N_RANDOM_ROWS = 10
REMOVE_DUPLICATE_SENTENCES = True

ACTIVE_FIELDS = SOURCE_FIELDS + ['sentence', 'voice', 'tense', 'adverb', 'subject',
                                 'subject_realized', 'subject_person', 'time_spec',
                                 'object_phrase_realized', 'placeholder_resolution']
PASSIVE_FIELDS = SOURCE_FIELDS + ['sentence', 'voice', 'tense', 'adverb', 'subject',
                                  'passive_person', 'object_gender', 'object_number',
                                  'time_spec', 'object_phrase_realized',
                                  'placeholder_resolution']
NOMINAL_FIELDS = SOURCE_FIELDS + ['sentence', 'voice', 'tense', 'adverb', 'time_spec',
                                  'subject', 'subject_base', 'subject_number',
                                  'subject_det', 'subject_det_type', 'subject_type',
                                  'person', 'context_gender', 'possessive',
                                  'object_phrase_realized', 'placeholder_resolution']


def read_csv(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as f:
        return [{k: (v.strip() if isinstance(v, str) else '')
                 for k, v in row.items()} for row in csv.DictReader(f)]


def distinct(values):
    return list(dict.fromkeys(x for x in values if x))


def load_inputs(directory=INPUT_DIR):
    directory = Path(directory)
    items = read_csv(directory / ITEMS_NAME)
    builders = read_csv(directory / BUILDER_NAME)
    inflections = read_csv(directory / VERBS_NAME)
    inflect = {(r['verb'], r['person']): r for r in inflections}
    if len(inflect) != len(inflections):
        raise ValueError('Duplicate verb/person pairs after disambiguation')
    for item in items:
        if any(not item.get(k) for k in SOURCE_FIELDS):
            raise ValueError('Incomplete item: ' + str(item))
        if item['gen_object_de'] not in ('M', 'F', 'N') or item['num_obj_de'] not in ('sg', 'pl'):
            raise ValueError('Invalid gender/number: ' + str(item))
        if (item['verb_de'], 'sie_pl') not in inflect:
            raise ValueError('Missing German conjugation: ' + item['verb_de'])
    return items, builders, inflect


def choose_context(builders, max_time_specs=MAX_TIME_SPECS, max_adverbs=MAX_ADVERBS,
                   adverb_selection=ADVERB_SELECTION):
    times = distinct(r['time_specs'] for r in builders)
    av = distinct(r['adverbs'] for r in builders)
    if adverb_selection is not None:
        missing = set(adverb_selection) - set(av)
        if missing:
            raise ValueError('Adverbs not in builder: ' + str(missing))
        av = [a for a in adverb_selection if a in av]
    return (times[:max_time_specs] if max_time_specs is not None else times,
            av[:max_adverbs] if max_adverbs is not None else av)


def accusative_indefinite(gender, number):
    if number == 'pl':
        raise ValueError('ART plural is unspecified; edit the source template')
    return {'M': 'einen', 'F': 'eine', 'N': 'ein'}[gender]


def resolve_object(item):
    """Retain source expression while resolving its small set of placeholders."""
    verb = item['verb_de']
    expression = item['full_expression']
    if not expression.endswith(' ' + verb):
        raise ValueError(f'Expected infinitive-final expression: {expression!r}')
    phrase = expression[:-(len(verb) + 1)].strip()
    placeholders = []
    if re.search(r'\bART\b', phrase):
        phrase = re.sub(r'\bART\b', accusative_indefinite(item['gen_object_de'], item['num_obj_de']), phrase)
        placeholders.append('ART->indefinite_accusative')
    if 'N-II' in phrase:
        # Specific source template: 'an N-II Rache nehmen'. This explicitly
        # realizes an oblique complement rather than deleting it silently.
        if item['object_de'] == 'Rache' and phrase == 'an N-II Rache':
            phrase = 'an jemandem Rache'
            placeholders.append('N-II->jemandem')
        else:
            raise ValueError(f'Unresolved complement placeholder in {expression}')
    if re.search(r'\b(?:ART|MOD|N-[IVX]+|N-\d+)\b', phrase):
        raise ValueError(f'Unresolved placeholder in {expression!r}')
    if item['object_de'] not in phrase:
        raise ValueError(f'Object {item["object_de"]!r} missing in {phrase!r}')
    return phrase, ';'.join(placeholders) if placeholders else 'none'


def to_nominative_np(item, object_phrase):
    """Convert a simple accusative object NP into passive nominative.

    Not a full German morphological parser: oblique complements must be handled
    separately and are omitted from passive generation by default.
    """
    gender, number = item['gen_object_de'], item['num_obj_de']
    if object_phrase.startswith('an jemandem '):
        raise ValueError('Oblique complement requires separate passive construction')
    if number == 'sg' and gender == 'M':
        for old, new in [('einen ', 'ein '), ('den ', 'der '),
                         ('diesen ', 'dieser '), ('jenen ', 'jener '),
                         ('seinen ', 'sein '), ('meinen ', 'mein ')]:
            if object_phrase.startswith(old):
                return new + object_phrase[len(old):]
    return object_phrase


def get_verb_parts(inflect, verb, person, tense, voice='active'):
    cols = TENSE_COL if voice == 'active' else PASSIVE_TENSE_COL
    key = cols[tense]
    record = inflect.get((verb, person))
    if not record or not record.get(key):
        raise ValueError(f'Missing inflection: {verb}, {person}, {key}')
    whole = record[key].strip()
    head, _, tail = whole.partition(' ')
    if not tail:
        raise ValueError(f'Expected compound tense but found {whole!r}')
    return head, tail


def make_active_sentence(time_spec, subject, finite, obj, tail, adverb='NA'):
    # Fronted time expression occupies position 1; finite verb is position 2.
    # No French-style comma after a fronted German adverbial.
    mid = ' '.join(x for x in (subject, '' if adverb == 'NA' else adverb, obj) if x)
    return f'{time_spec} {finite} {mid} {tail}.'.replace('  ', ' ')


def make_passive_sentence(time_spec, subject_np, finite, tail, adverb='NA'):
    # Passive adverb before nominal subject is more idiomatic in many contexts.
    mid = ' '.join(x for x in ('' if adverb == 'NA' else adverb, subject_np) if x)
    return f'{time_spec} {finite} {mid} {tail}.'.replace('  ', ' ')


def csv_writer(path, fields):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    f = p.open('w', encoding='utf-8-sig', newline='')
    w = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
    w.writeheader()
    return f, w


def generate_active_pronominal(items, builders, inflect, output_dir=OUTPUT_DIR,
                               pronouns=ACTIVE_PRONOUNS, tenses=TENSES,
                               max_time_specs=MAX_TIME_SPECS, max_adverbs=MAX_ADVERBS,
                               adverb_selection=ADVERB_SELECTION):
    times, adverbs = choose_context(builders, max_time_specs, max_adverbs, adverb_selection)
    output_dir = Path(output_dir) / 'DE_active_passive'
    records = []
    result = {}
    for tense in tenses:
        for condition in ['no_adverbs', 'with_adverbs']:
            filename = output_dir / f'LVC_DE_active_{tense}_{condition}.csv'
            f, w = csv_writer(filename, ACTIVE_FIELDS)
            count = 0
            with f:
                for item in items:
                    obj, resolution = resolve_object(item)
                    for t in times:
                        for person in pronouns:
                            if person not in {'ich','du','er','sie_sg','es','wir','ihr','sie_pl','Sie_formal'}:
                                raise ValueError(f'Unknown person: {person}')
                            finite, tail = get_verb_parts(inflect,item['verb_de'],person,tense)
                            subj = {'sie_sg':'sie', 'sie_pl':'sie', 'Sie_formal':'Sie'}.get(person, person)
                            for adv in (['NA'] if condition == 'no_adverbs' else adverbs):
                                r = dict(item)
                                r.update(sentence=make_active_sentence(t,subj,finite,obj,tail,adv),
                                         voice='active',tense=tense,adverb=adv,
                                         subject=subj,subject_realized='yes',
                                         subject_person=person,time_spec=t,
                                         object_phrase_realized=obj,placeholder_resolution=resolution)
                                w.writerow(r); count += 1
            result[filename.name] = count
            records.append(str(filename))
    return result


def passive_exclusion_reason(item):
    if item['verb_de'] in PASSIVE_EXCLUDED_VERBS:
        return 'haben does not permit ordinary personal event passive'
    if item['object_de'] == 'Rache' and item['verb_de'] == 'nehmen':
        return 'oblique template an N-II Rache nehmen needs separate impersonal-passive handling'
    return ''


def generate_passive(items, builders, inflect, output_dir=OUTPUT_DIR,
                     tenses=TENSES, max_time_specs=MAX_TIME_SPECS,
                     max_adverbs=MAX_ADVERBS, adverb_selection=ADVERB_SELECTION):
    times, adverbs = choose_context(builders,max_time_specs,max_adverbs,adverb_selection)
    output_dir = Path(output_dir) / 'DE_active_passive'
    excluded = []
    included = []
    for item in items:
        reason = passive_exclusion_reason(item)
        if reason:
            excluded.append(dict(item,skip_reason=reason))
        else:
            included.append(item)
    excluded_path = output_dir / 'LVC_DE_passive_excluded_items.csv'
    f,w = csv_writer(excluded_path,SOURCE_FIELDS+['skip_reason'])
    with f: w.writerows(excluded)
    result = {}
    for tense in tenses:
        for condition in ['no_adverbs','with_adverbs']:
            filename=output_dir / f'LVC_DE_passive_{tense}_{condition}.csv'
            f,w=csv_writer(filename,PASSIVE_FIELDS)
            count=0
            with f:
                for item in included:
                    obj,resolution=resolve_object(item)
                    passive_subject=to_nominative_np(item,obj)
                    person='sie_pl' if item['num_obj_de']=='pl' else {'M':'er','F':'sie_sg','N':'es'}[item['gen_object_de']]
                    finite,tail=get_verb_parts(inflect,item['verb_de'],person,tense,voice='passive')
                    for t in times:
                        for adv in (['NA'] if condition=='no_adverbs' else adverbs):
                            r=dict(item)
                            r.update(sentence=make_passive_sentence(t,passive_subject,finite,tail,adv),
                                     voice='passive',tense=tense,adverb=adv,subject=passive_subject,
                                     passive_person=person,object_gender=item['gen_object_de'],
                                     object_number=item['num_obj_de'],time_spec=t,
                                     object_phrase_realized=obj,placeholder_resolution=resolution)
                            w.writerow(r);count+=1
            result[filename.name]=count
    result['excluded_items']=len(excluded)
    result['eligible_items']=len(included)
    return result

# Substantivized adjectives require special inflection after determiners.
ADJECTIVAL_NOUNS = {
    'Büroangestellter','Büroangestellte','Angestellter','Angestellte',
    'Überlebender','Überlebende','Reisender','Reisende','Freiwilliger','Freiwillige',
}


def inflect_nominal_subject(noun, gender, number, det):
    if noun not in ADJECTIVAL_NOUNS:
        return noun
    base = re.sub(r'(er|e|en)$','',noun)
    if number == 'pl':
        # die/meine Angestellten, einige Angestellte
        ending = 'e' if det in {'einige','mehrere','viele','wenige'} else 'en'
        return base + ending
    if gender == 'F':
        return base + 'e'       # eine/die Angestellte
    if gender == 'M':
        # der/dieser Angestellte vs ein/mein Angestellter
        ending = 'er' if det in {'ein','mein','dein','sein','ihr','unser','euer'} else 'e'
        return base + ending
    return noun


def subject_candidates(builders, subject_number=SUBJECT_NUMBER,
                       det_types=SUBJECT_DET_TYPES,
                       use_nominal=USE_NOMINAL_SUBJECTS,
                       use_pronouns=USE_PRONOUN_SUBJECTS):
    if subject_number not in ('sg','pl','both'):
        raise ValueError('SUBJECT_NUMBER must be sg, pl or both')
    for dt in det_types:
        if dt not in ('det','det_poss','no_det'):
            raise ValueError(f'Unknown determiner type {dt!r}')
    col_lists = {key: distinct(row.get(key,'') for row in builders)
                 for key in ('det_masc','det_fem','det_neut','det_pl','quant',
                             'poss_masc','poss_fem','poss_neut','poss_pl')}
    out = []
    if use_nominal:
        for row in builders:
            noun = row.get('subject_sg','')
            plural = row.get('subject_pl_noun','')
            gender = row.get('gender_sbj','').upper()
            poss = row.get('det_poss_tf','').upper()
            if gender not in ('M','F','N'):
                if noun: raise ValueError(f'Invalid subject gender for {noun}: {gender}')
                continue
            for number,base in [('sg',noun),('pl',plural)]:
                if not base or (subject_number!='both' and subject_number !=number):
                    continue
                person = ('sie_pl' if number=='pl' else {'M':'er','F':'sie_sg','N':'es'}[gender])
                gender_context = gender if number=='sg' else 'PL'
                if number=='sg':
                    ordinary=col_lists[{'M':'det_masc','F':'det_fem','N':'det_neut'}[gender]]
                    possessive=col_lists[{'M':'poss_masc','F':'poss_fem','N':'poss_neut'}[gender]]
                else:
                    ordinary=col_lists['det_pl'] + col_lists['quant']
                    possessive=col_lists['poss_pl']
                alternatives=[]
                if 'det' in det_types:
                    alternatives.extend((d,'det') for d in ordinary)
                if 'det_poss' in det_types and poss=='T':
                    alternatives.extend((d,'det_poss') for d in possessive)
                if 'no_det' in det_types and number=='pl':
                    alternatives.append(('NA','no_det'))
                # Singular bare count nouns are intentionally not produced.
                for det,kind in alternatives:
                    inflected=inflect_nominal_subject(base,gender,number,det)
                    full=(det+' '+inflected).strip() if det!='NA' else inflected
                    out.append(dict(subject=full,subject_base=base,subject_number=number,
                                    subject_det=det,subject_det_type=kind,subject_type='nominal',
                                    person=person,context_gender=gender_context,possessive=poss))
    if use_pronouns:
        for person in ['ich','du','er','sie_sg','es','wir','ihr','sie_pl']:
            number='pl' if person in ['wir','ihr','sie_pl'] else 'sg'
            if subject_number!='both' and subject_number != number:
                continue
            surface='sie' if person in ('sie_sg','sie_pl') else person
            out.append(dict(subject=surface,subject_base=surface,subject_number=number,
                            subject_det='NA',subject_det_type='NA',subject_type='pronoun',
                            person=person,context_gender='NA',possessive='NA'))
    # Multiple builder rows can repeat same noun; remove duplicate combinations.
    return list({tuple(sorted(r.items())):r for r in out}.values())


def generate_active_nominal(items, builders, inflect, output_dir=OUTPUT_DIR,
                            subject_number=SUBJECT_NUMBER,det_types=SUBJECT_DET_TYPES,
                            use_pronouns=USE_PRONOUN_SUBJECTS,use_nominal=USE_NOMINAL_SUBJECTS,
                            tenses=NOMINAL_TENSES,use_adverbs=USE_ADVERBS_NOMINAL,
                            random_subject_selection=RANDOM_SUBJECT_SELECTION,
                            random_output_sample=RANDOM_OUTPUT_SAMPLE,
                            n_random_rows=N_RANDOM_ROWS,random_state=RANDOM_STATE,
                            remove_duplicate_sentences=REMOVE_DUPLICATE_SENTENCES,
                            max_time_specs=MAX_TIME_SPECS,max_adverbs=MAX_ADVERBS,
                            adverb_selection=ADVERB_SELECTION):
    rng=random.Random(random_state)
    subj=subject_candidates(builders,subject_number,det_types,use_nominal,use_pronouns)
    if not subj: raise ValueError('No nominal/pronominal subjects after filtering')
    times,adverbs=choose_context(builders,max_time_specs,max_adverbs,adverb_selection)
    av=adverbs if use_adverbs else ['NA']
    result=[];seen=set()
    for item in items:
        obj,resolution=resolve_object(item)
        for t in times:
            for adv in av:
                for tense in tenses:
                    options=[rng.choice(subj)] if random_subject_selection else subj
                    for s in options:
                        finite,tail=get_verb_parts(inflect,item['verb_de'],s['person'],tense)
                        sentence=make_active_sentence(t,s['subject'],finite,obj,tail,adv)
                        if remove_duplicate_sentences and sentence in seen: continue
                        seen.add(sentence)
                        r=dict(item);r.update(s)
                        r.update(sentence=sentence,voice='active',tense=tense,adverb=adv,
                                 time_spec=t,object_phrase_realized=obj,
                                 placeholder_resolution=resolution)
                        result.append(r)
    count_before_sample=len(result)
    if random_output_sample and len(result)>n_random_rows:
        result=rng.sample(result,n_random_rows)
    target=Path(output_dir) / 'DE_active_nominal_subjects' / 'LVC_DE_active_nominal_subjects_output.csv'
    f,w=csv_writer(target,NOMINAL_FIELDS)
    with f: w.writerows(result)
    return {'output_file':str(target),'available_subjects':len(subj),
            'generated_before_sampling':count_before_sample,'saved_rows':len(result)}


def generate_all(items, builders, inflect, output_dir=OUTPUT_DIR):
    out = Path(output_dir)
    result = {
        'active_pronominal': generate_active_pronominal(items,builders,inflect,out),
        'passive':generate_passive(items,builders,inflect,out),
        'active_nominal':generate_active_nominal(items,builders,inflect,out),
    }
    out.mkdir(parents=True,exist_ok=True)
    with (out/'generation_report_DE.json').open('w',encoding='utf-8') as f:
        json.dump(result,f,indent=2,ensure_ascii=False)
    return result



items, builder, verbs = load_inputs(INPUT_DIR)
print(f"Loaded {len(items)} items, {len(builder)} builder rows, and {len(verbs)} verb/person forms")
print("Default pronominal subjects:", ACTIVE_PRONOUNS)


## 1. Active sentences — pronominal subjects

Six CSV files, by tense and adverb condition, exactly as in the French active cell. Default subject is third-person plural *sie*. Change `ACTIVE_PRONOUNS` for other person labels such as `ich`, `er`, `sie_sg`, `sie_pl`.


In [ ]:
active_counts = generate_active_pronominal(
    items, builder, verbs,
    output_dir=OUTPUT_DIR,
    pronouns=ACTIVE_PRONOUNS,
    tenses=TENSES,
    max_time_specs=MAX_TIME_SPECS,
    max_adverbs=MAX_ADVERBS,
    adverb_selection=ADVERB_SELECTION,
)
for name, count in active_counts.items():
    print(f'{name}: {count:,} sentences')


## 2. Passive sentences

Six CSV files, plus an explicit list of excluded items. Accusative direct objects become nominative passive subjects; agreement is based on `gen_object_de` and `num_obj_de`. German *haben* does not permit an ordinary personal event passive, and `an N-II Rache nehmen` needs a different impersonal template, so both are excluded by default.


In [ ]:
passive_counts = generate_passive(
    items, builder, verbs,
    output_dir=OUTPUT_DIR,
    tenses=TENSES,
    max_time_specs=MAX_TIME_SPECS,
    max_adverbs=MAX_ADVERBS,
    adverb_selection=ADVERB_SELECTION,
)
for name, count in passive_counts.items():
    print(f'{name}: {count:,}')


## 3. Active sentences — nominal subjects

Preserves the French v4 switches: singular/plural/both, determiner types, nominal vs. pronoun subjects, optional adverbs, tense selection, reproducible random subject selection, 10-row output sampling, and duplicate removal. For common nouns, the default `SUBJECT_DET_TYPES = ["det"]` prevents ungrammatical bare singular noun phrases.


In [ ]:
nominal_result = generate_active_nominal(
    items, builder, verbs,
    output_dir=OUTPUT_DIR,
    subject_number=SUBJECT_NUMBER,
    det_types=SUBJECT_DET_TYPES,
    use_pronouns=USE_PRONOUN_SUBJECTS,
    use_nominal=USE_NOMINAL_SUBJECTS,
    tenses=NOMINAL_TENSES,
    use_adverbs=USE_ADVERBS_NOMINAL,
    random_subject_selection=RANDOM_SUBJECT_SELECTION,
    random_output_sample=RANDOM_OUTPUT_SAMPLE,
    n_random_rows=N_RANDOM_ROWS,
    random_state=RANDOM_STATE,
    remove_duplicate_sentences=REMOVE_DUPLICATE_SENTENCES,
    max_time_specs=MAX_TIME_SPECS,
    max_adverbs=MAX_ADVERBS,
    adverb_selection=ADVERB_SELECTION,
)
print(json.dumps(nominal_result, indent=2, ensure_ascii=False))


## 4. Basic validation examples

Simple assertions for German verb-second order, article case and ambiguous *sie* forms. This cell does not alter the input files.


In [ ]:
assert make_active_sentence('Anfang Januar', 'sie', 'haben', 'einen Brief', 'gegeben') == 'Anfang Januar haben sie einen Brief gegeben.'
assert make_active_sentence('Anfang Januar', 'sie', 'werden', 'ein Paket', 'erhalten') == 'Anfang Januar werden sie ein Paket erhalten.'
assert make_passive_sentence('Anfang Januar', 'ein Brief', 'ist', 'gegeben worden') == 'Anfang Januar ist ein Brief gegeben worden.'
assert to_nominative_np({'gen_object_de':'M','num_obj_de':'sg'},'einen Brief') == 'ein Brief'
assert get_verb_parts(verbs,'geben','sie_sg','present_perfect') == ('hat','gegeben')
assert get_verb_parts(verbs,'geben','sie_pl','present_perfect') == ('haben','gegeben')
assert 'der Angestellte' in [x['subject'] for x in subject_candidates(builder) if x['subject_base']=='Angestellter']
print('German sentence-generation checks passed.')
